<!--nav--> [🗺 Learning path](README.md) · **19/41** · ◀ [Simple MultiGPU Diffusion](./Simple_MultiGPU_Diffusion.ipynb) · [Simple MultiGPU Audio](./Simple_MultiGPU_Audio.ipynb) ▶

# Simple Multi-GPU: Image Classification with ViT

Teach a Vision Transformer to classify images.

- **Model:** ViT-base (86M params) — Google's Vision Transformer
- **Method:** LoRA on attention layers + DeepSpeed ZeRO-2
- **Data:** Food-101 (101 food categories, 1000 examples)
- **Platform:** Kaggle 2x T4 (free) or Colab T4 (free)

### How ViT works (30 seconds)

```
Image (224x224) ──> Split into 16x16 patches (196 patches)
                         |
                    Each patch = a "token" (like a word)
                         |
                    Transformer encoder (same as BERT)
                         |
                    [CLS] token ──> Classification head ──> "pizza"
```

It's literally a text transformer that reads image patches instead of words.

## Step 1: Install

In [ ]:
!pip install -q transformers datasets peft accelerate deepspeed evaluate scikit-learn

## Step 2: Detect GPUs

In [ ]:
import torch, os, json, time

assert torch.cuda.is_available(), "GPU required!"

NUM_GPUS = torch.cuda.device_count()
for i in range(NUM_GPUS):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {name} ({mem:.0f} GB)")
print(f"\nTotal GPUs: {NUM_GPUS}")

## Step 3: Write Config

Accelerate config with inline DeepSpeed ZeRO-2 setup.

In [ ]:
# Accelerate config — accelerate-managed DeepSpeed
accel_yaml = f"""compute_environment: LOCAL_MACHINE
distributed_type: DEEPSPEED
deepspeed_config:
  gradient_accumulation_steps: auto
  gradient_clipping: auto
  offload_optimizer_device: cpu
  offload_param_device: none
  zero3_init_flag: false
  zero_stage: 2
machine_rank: 0
main_process_ip: null
main_process_port: null
main_training_function: main
mixed_precision: bf16
num_machines: 1
num_processes: {NUM_GPUS}
use_cpu: false
"""
accel_dir = os.path.expanduser("~/.cache/huggingface/accelerate")
os.makedirs(accel_dir, exist_ok=True)
with open(os.path.join(accel_dir, "default_config.yaml"), "w") as f:
    f.write(accel_yaml)

print(f"Config written. {NUM_GPUS} GPU(s), ZeRO-2, bf16.")

## Step 4: Preview the Data

Food-101: 101 categories of food photos. We'll use a small subset for speed.

In [ ]:
from datasets import load_dataset
from IPython.display import display

# Load a small preview
preview = load_dataset("food101", split="train", streaming=True)
examples = []
for i, ex in enumerate(preview):
    if i >= 6:
        break
    examples.append(ex)

# Get label names
ds_info = load_dataset("food101", split="train[:1]")
label_names = ds_info.features["label"].names
print(f"Total categories: {len(label_names)}")
print(f"Examples: {label_names[:10]}...\n")

for ex in examples[:3]:
    print(f"Label: {label_names[ex['label']]}")
    display(ex["image"].resize((200, 200)))

## Step 5: Write Training Script

Every step explained:

1. Load ViT-base (pretrained on ImageNet)
2. Add a classification head (101 food classes)
3. Add LoRA to the attention layers
4. Load images, resize to 224x224, normalize
5. Train — model sees image, predicts food category, learns from mistakes

In [ ]:
%%writefile train_vit.py
"""Distributed image classification: ViT-base + LoRA + DeepSpeed."""
import torch, os, json, time
import numpy as np
os.environ["WANDB_DISABLED"] = "true"

from datasets import load_dataset
from transformers import (
    ViTForImageClassification,  # ViT with a classification head on top
    ViTImageProcessor,          # Resizes + normalizes images for ViT
    TrainingArguments,
    Trainer,
    TrainerCallback,
)
from peft import LoraConfig, get_peft_model

MODEL = "google/vit-base-patch16-224-in21k"  # Pretrained on ImageNet-21k
NUM_CLASSES = 101  # Food-101 categories

# ── Step 1: Load the image processor ────────────────────
# This handles: resize to 224x224, normalize pixel values
processor = ViTImageProcessor.from_pretrained(MODEL)

# ── Step 2: Load ViT + add classification head ──────────
# ViT-base was pretrained for general image understanding.
# We add a fresh linear layer on top: 768 hidden → 101 food classes
model = ViTForImageClassification.from_pretrained(
    MODEL,
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True,  # The pretrained head has different size — that's fine
    dtype=torch.bfloat16,
)
total_params = sum(p.numel() for p in model.parameters())

# ── Step 3: Add LoRA ────────────────────────────────────
# ViT's transformer has the same q/v projections as language models.
# LoRA trains ~0.7% of parameters instead of all 86M.
model = get_peft_model(model, LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query", "value"],  # ViT attention layers
    bias="none",
    modules_to_save=["classifier"],  # Also train the new classification head
))
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
model.print_trainable_parameters()

# ── Step 4: Load Food-101 dataset ───────────────────────
# 1000 train + 200 eval examples (small for demo speed)
train_ds = load_dataset("food101", split="train")
train_ds = train_ds.shuffle(seed=42).select(range(1000))

eval_ds = load_dataset("food101", split="validation")
eval_ds = eval_ds.shuffle(seed=42).select(range(200))

label_names = load_dataset("food101", split="train[:1]").features["label"].names
print(f"Train: {len(train_ds)} | Eval: {len(eval_ds)} | Classes: {len(label_names)}")

# ── Step 5: Preprocess images ───────────────────────────
# Convert PIL images → tensors that ViT expects
def preprocess(batch):
    images = [img.convert("RGB") for img in batch["image"]]
    inputs = processor(images, return_tensors="pt")
    inputs["labels"] = batch["label"]
    return inputs

train_ds = train_ds.with_transform(preprocess)
eval_ds = eval_ds.with_transform(preprocess)


# ── Step 6: Custom collator ─────────────────────────────
# Stack images and labels into batches
def collate_fn(batch):
    return {
        "pixel_values": torch.stack([x["pixel_values"] for x in batch]),
        "labels": torch.tensor([x["labels"] for x in batch]),
    }


# ── Step 7: Metrics ─────────────────────────────────────
# Accuracy: what % of images did the model classify correctly?
def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    labels = eval_pred.label_ids
    accuracy = (predictions == labels).mean()
    return {"accuracy": accuracy}


# ── Metrics callback (same as text notebook) ────────────
class MetricsCallback(TrainerCallback):
    def __init__(self):
        self.logs = []
        self.step_times = []
        self.train_start = None
        self.eval_results = []

    def on_train_begin(self, args, state, control, **kwargs):
        self.train_start = time.time()
        torch.cuda.reset_peak_memory_stats()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            self.logs.append({
                "step": state.global_step,
                "loss": logs["loss"],
                "learning_rate": logs.get("learning_rate", 0),
                "epoch": logs.get("epoch", 0),
            })
        if logs and "eval_accuracy" in logs:
            self.eval_results.append({
                "step": state.global_step,
                "accuracy": logs["eval_accuracy"],
                "eval_loss": logs.get("eval_loss", 0),
            })

    def on_step_end(self, args, state, control, **kwargs):
        self.step_times.append(time.time())

    def on_train_end(self, args, state, control, **kwargs):
        if int(os.environ.get("LOCAL_RANK", 0)) != 0:
            return
        train_time = time.time() - self.train_start
        gpu_mem_alloc = torch.cuda.max_memory_allocated() / 1e9
        gpu_mem_res = torch.cuda.max_memory_reserved() / 1e9
        num_gpus = int(os.environ.get("WORLD_SIZE", 1))
        total_steps = state.global_step
        total_samples = total_steps * args.per_device_train_batch_size * args.gradient_accumulation_steps * num_gpus

        if len(self.step_times) > 2:
            durations = [self.step_times[i+1] - self.step_times[i] for i in range(len(self.step_times)-1)]
            avg_step = sum(durations) / len(durations)
        else:
            avg_step = train_time / max(total_steps, 1)

        best_acc = max((e["accuracy"] for e in self.eval_results), default=0)

        metrics = {
            "model": MODEL,
            "task": "image-classification",
            "num_classes": NUM_CLASSES,
            "total_params": total_params,
            "trainable_params": trainable_params,
            "trainable_pct": round(trainable_params / total_params * 100, 2),
            "num_gpus": num_gpus,
            "gpu_name": torch.cuda.get_device_name(0),
            "gpu_mem_allocated_gb": round(gpu_mem_alloc, 2),
            "gpu_mem_reserved_gb": round(gpu_mem_res, 2),
            "gpu_mem_total_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
            "train_time_sec": round(train_time, 1),
            "total_steps": total_steps,
            "total_images": total_samples,
            "images_per_sec": round(total_samples / train_time, 2),
            "avg_step_time_sec": round(avg_step, 3),
            "final_loss": self.logs[-1]["loss"] if self.logs else None,
            "best_accuracy": round(best_acc, 4),
            "loss_history": self.logs,
            "eval_history": self.eval_results,
            "batch_size_per_gpu": args.per_device_train_batch_size,
            "grad_accum_steps": args.gradient_accumulation_steps,
            "effective_batch_size": args.per_device_train_batch_size * args.gradient_accumulation_steps * num_gpus,
            "learning_rate": args.learning_rate,
        }
        with open("training_metrics.json", "w") as f:
            json.dump(metrics, f, indent=2)
        print(f"\nBest accuracy: {best_acc:.1%}")
        print(f"Metrics saved to training_metrics.json")


metrics_cb = MetricsCallback()

# ── Step 8: Train ───────────────────────────────────────
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./vit_output",
        num_train_epochs=5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=2,
        learning_rate=5e-4,
        warmup_ratio=0.1,
        logging_steps=1,
        eval_strategy="epoch",
        bf16=True,
        gradient_checkpointing=True,
        report_to="none",
        save_strategy="no",
        remove_unused_columns=False,
    ),
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    callbacks=[metrics_cb],
)

trainer.train()
trainer.save_model("./vit_output/final")
processor.save_pretrained("./vit_output/final")

if int(os.environ.get("LOCAL_RANK", 0)) == 0:
    print("\n" + "=" * 50)
    print("Image classification training complete!")
    print("=" * 50)

## Step 6: Launch Training

In [ ]:
print(f"Launching on {NUM_GPUS} GPU(s)...")
start = time.time()

!accelerate launch --num_processes={NUM_GPUS} train_vit.py

elapsed = time.time() - start
print(f"\nTotal time: {elapsed:.0f}s")

## Step 7: Performance Dashboard

In [ ]:
import json, matplotlib.pyplot as plt, matplotlib.ticker as ticker
from IPython.display import HTML, display

with open("training_metrics.json") as f:
    m = json.load(f)

steps = [h["step"] for h in m["loss_history"]]
losses = [h["loss"] for h in m["loss_history"]]
lrs = [h["learning_rate"] for h in m["loss_history"]]

# --- Loss + Accuracy (dual plot) ---
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(14, 4))
fig.patch.set_facecolor("#0d1117")

# Left: Training loss
ax_loss.set_facecolor("#0d1117")
ax_loss.plot(steps, losses, color="#58a6ff", linewidth=2)
ax_loss.fill_between(steps, losses, alpha=0.1, color="#58a6ff")
ax_loss.set_xlabel("Step", color="#8b949e", fontsize=11)
ax_loss.set_ylabel("Loss", color="#58a6ff", fontsize=11)
ax_loss.set_title("Training Loss", color="#e6edf3", fontsize=13, fontweight="bold")
ax_loss.tick_params(colors="#8b949e")
ax_loss.grid(True, alpha=0.15, color="#30363d")
for spine in ax_loss.spines.values():
    spine.set_color("#30363d")

# Right: Eval accuracy per epoch
ax_acc.set_facecolor("#0d1117")
if m["eval_history"]:
    eval_steps = [e["step"] for e in m["eval_history"]]
    accs = [e["accuracy"] * 100 for e in m["eval_history"]]
    ax_acc.plot(range(1, len(accs) + 1), accs, color="#3fb950", linewidth=2.5, marker="o", markersize=8)
    ax_acc.fill_between(range(1, len(accs) + 1), accs, alpha=0.1, color="#3fb950")
    for i, acc in enumerate(accs):
        ax_acc.annotate(f"{acc:.1f}%", (i + 1, acc), textcoords="offset points",
                        xytext=(0, 12), ha="center", color="#3fb950", fontweight="bold", fontsize=11)
ax_acc.set_xlabel("Epoch", color="#8b949e", fontsize=11)
ax_acc.set_ylabel("Accuracy %", color="#3fb950", fontsize=11)
ax_acc.set_title("Eval Accuracy", color="#e6edf3", fontsize=13, fontweight="bold")
ax_acc.tick_params(colors="#8b949e")
ax_acc.grid(True, alpha=0.15, color="#30363d")
for spine in ax_acc.spines.values():
    spine.set_color("#30363d")

plt.tight_layout()
plt.show()

# --- GPU Memory ---
fig2, ax3 = plt.subplots(figsize=(6, 3))
fig2.patch.set_facecolor("#0d1117")
ax3.set_facecolor("#0d1117")
mem_labels = ["Allocated", "Reserved", "Total"]
mem_vals = [m["gpu_mem_allocated_gb"], m["gpu_mem_reserved_gb"], m["gpu_mem_total_gb"]]
colors = ["#3fb950", "#58a6ff", "#30363d"]
bars = ax3.barh(mem_labels, mem_vals, color=colors, height=0.5, edgecolor="#0d1117")
for bar, val in zip(bars, mem_vals):
    ax3.text(val + 0.1, bar.get_y() + bar.get_height()/2, f"{val:.1f} GB",
             va="center", color="#e6edf3", fontsize=11, fontweight="bold")
ax3.set_xlim(0, m["gpu_mem_total_gb"] * 1.3)
ax3.set_title(f"GPU Memory — {m['gpu_name']}", color="#e6edf3", fontsize=13, fontweight="bold")
ax3.tick_params(colors="#8b949e")
ax3.spines["top"].set_visible(False)
ax3.spines["right"].set_visible(False)
for spine in ax3.spines.values():
    spine.set_color("#30363d")
plt.tight_layout()
plt.show()

# --- HTML Dashboard ---
loss_drop = ""
if len(losses) >= 2:
    pct = (losses[0] - losses[-1]) / losses[0] * 100
    loss_drop = f"{pct:.0f}% drop"

mem_util = m["gpu_mem_allocated_gb"] / m["gpu_mem_total_gb"] * 100

html = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
            max-width: 780px; margin: 20px 0;">

  <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin-bottom: 16px;">
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 16px; text-align: center;">
      <div style="color: #8b949e; font-size: 11px; text-transform: uppercase; letter-spacing: 1px;">Best Accuracy</div>
      <div style="color: #3fb950; font-size: 26px; font-weight: 700; margin: 6px 0;">{m['best_accuracy']*100:.1f}%</div>
      <div style="color: #8b949e; font-size: 12px;">{m['num_classes']} classes</div>
    </div>
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 16px; text-align: center;">
      <div style="color: #8b949e; font-size: 11px; text-transform: uppercase; letter-spacing: 1px;">Throughput</div>
      <div style="color: #58a6ff; font-size: 26px; font-weight: 700; margin: 6px 0;">{m['images_per_sec']:.1f}</div>
      <div style="color: #8b949e; font-size: 12px;">images/sec</div>
    </div>
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 16px; text-align: center;">
      <div style="color: #8b949e; font-size: 11px; text-transform: uppercase; letter-spacing: 1px;">Final Loss</div>
      <div style="color: #f0883e; font-size: 26px; font-weight: 700; margin: 6px 0;">{m['final_loss']:.3f}</div>
      <div style="color: #8b949e; font-size: 12px;">{loss_drop}</div>
    </div>
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 16px; text-align: center;">
      <div style="color: #8b949e; font-size: 11px; text-transform: uppercase; letter-spacing: 1px;">Train Time</div>
      <div style="color: #d2a8ff; font-size: 26px; font-weight: 700; margin: 6px 0;">{m['train_time_sec']:.0f}s</div>
      <div style="color: #8b949e; font-size: 12px;">{m['total_steps']} steps</div>
    </div>
  </div>

  <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 12px;">
    <div style="background: #161b22; border: 1px solid #30363d; border-radius: 12px; padding: 18px;">
      <div style="color: #e6edf3; font-size: 13px; font-weight: 600; margin-bottom: 12px;
                  border-bottom: 1px solid #21262d; padding-bottom: 8px;">Model & Training</div>
      <table style="width:100%; color: #c9d1d9; font-size: 12px; border-spacing: 0 6px;">
        <tr><td style="color:#8b949e;">Model</td><td style="text-align:right; font-weight:600;">ViT-base-patch16</td></tr>
        <tr><td style="color:#8b949e;">Total params</td><td style="text-align:right;">{m['total_params']/1e6:.0f}M</td></tr>
        <tr><td style="color:#8b949e;">Trainable (LoRA + head)</td>
            <td style="text-align:right; color:#3fb950;">{m['trainable_params']/1e3:.0f}K ({m['trainable_pct']:.2f}%)</td></tr>
        <tr><td style="color:#8b949e;">Effective batch</td>
            <td style="text-align:right;">{m['batch_size_per_gpu']} x {m['grad_accum_steps']} x {m['num_gpus']}GPU = {m['effective_batch_size']}</td></tr>
        <tr><td style="color:#8b949e;">Learning rate</td><td style="text-align:right;">{m['learning_rate']}</td></tr>
        <tr><td style="color:#8b949e;">Precision</td><td style="text-align:right;">bf16</td></tr>
      </table>
    </div>
    <div style="background: #161b22; border: 1px solid #30363d; border-radius: 12px; padding: 18px;">
      <div style="color: #e6edf3; font-size: 13px; font-weight: 600; margin-bottom: 12px;
                  border-bottom: 1px solid #21262d; padding-bottom: 8px;">GPU & Memory</div>
      <table style="width:100%; color: #c9d1d9; font-size: 12px; border-spacing: 0 6px;">
        <tr><td style="color:#8b949e;">GPUs</td><td style="text-align:right; font-weight:600;">{m['num_gpus']}x {m['gpu_name']}</td></tr>
        <tr><td style="color:#8b949e;">VRAM total</td><td style="text-align:right;">{m['gpu_mem_total_gb']} GB per GPU</td></tr>
        <tr><td style="color:#8b949e;">Peak allocated</td>
            <td style="text-align:right; color:#3fb950;">{m['gpu_mem_allocated_gb']} GB ({mem_util:.0f}%)</td></tr>
        <tr><td style="color:#8b949e;">Peak reserved</td><td style="text-align:right;">{m['gpu_mem_reserved_gb']} GB</td></tr>
        <tr><td style="color:#8b949e;">Optimizer</td><td style="text-align:right;">CPU offload (ZeRO-2)</td></tr>
        <tr><td style="color:#8b949e;">Avg step time</td><td style="text-align:right;">{m['avg_step_time_sec']*1000:.0f} ms</td></tr>
      </table>
    </div>
  </div>

  <div style="background: #161b22; border: 1px solid #30363d; border-radius: 12px; padding: 14px 18px;
              margin-top: 12px; display: flex; justify-content: space-between; align-items: center;">
    <span style="color: #8b949e; font-size: 12px;">GPU Memory Utilization</span>
    <div style="flex: 1; margin: 0 16px; background: #21262d; border-radius: 6px; height: 18px; overflow: hidden;">
      <div style="width: {mem_util:.0f}%; height: 100%; border-radius: 6px;
                  background: linear-gradient(90deg, #238636, #3fb950);"></div>
    </div>
    <span style="color: #3fb950; font-size: 13px; font-weight: 700;">{mem_util:.0f}%</span>
  </div>

</div>
"""
display(HTML(html))

## Step 8: Test — Classify Some Food Photos

In [ ]:
import torch
from transformers import ViTForImageClassification, ViTImageProcessor
from peft import PeftModel
from datasets import load_dataset
from IPython.display import display

# Load model
base_model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224-in21k",
    num_labels=101, ignore_mismatched_sizes=True,
    dtype=torch.bfloat16,
).to("cuda")
model = PeftModel.from_pretrained(base_model, "./vit_output/final")
model.eval()
processor = ViTImageProcessor.from_pretrained("./vit_output/final")

# Get label names
label_names = load_dataset("food101", split="train[:1]").features["label"].names

# Grab some test images
test_ds = load_dataset("food101", split="validation")
test_ds = test_ds.shuffle(seed=123).select(range(8))

correct = 0
for ex in test_ds:
    image = ex["image"].convert("RGB")
    true_label = label_names[ex["label"]]

    inputs = processor(image, return_tensors="pt").to("cuda")
    with torch.no_grad():
        logits = model(**inputs).logits

    pred_idx = logits.argmax(-1).item()
    pred_label = label_names[pred_idx]
    confidence = torch.softmax(logits, dim=-1)[0, pred_idx].item()
    is_correct = pred_label == true_label
    correct += is_correct

    status = "correct" if is_correct else f"WRONG (true: {true_label})"
    print(f"\nPredicted: {pred_label} ({confidence:.0%}) — {status}")
    display(image.resize((200, 200)))

print(f"\n{'='*40}")
print(f"Got {correct}/{len(test_ds)} correct ({correct/len(test_ds):.0%})")

---

## How It All Works

### ViT architecture

```
Image (224 x 224 x 3)
        |
  Split into 196 patches (16x16 each)
        |
  Each patch → linear projection → "token" (768-dim)
        |
  [CLS] + 196 patch tokens + position embeddings
        |
  12 Transformer encoder layers  ← LoRA goes here (query + value)
        |
  [CLS] output → Linear(768, 101) → softmax → "pizza"
                       ↑
              Classification head (also trained)
```

### What's trained vs frozen

| Component | Params | Trained? |
|-----------|--------|----------|
| Patch embeddings | 590K | Frozen |
| Transformer layers (base) | 85M | Frozen |
| LoRA on query + value | ~590K | **Trained** |
| Classification head (101 classes) | ~77K | **Trained** |

### Comparison with other notebooks

| Notebook | Input | Model | Task | Output |
|----------|-------|-------|------|--------|
| Text (GPT-2) | Words | Decoder | Next token | Text |
| Audio (Whisper) | Sound | Enc-Dec | Transcribe | Text |
| Diffusion (SD 1.5) | Text | UNet | Denoise | Image |
| **This (ViT)** | **Image** | **Encoder** | **Classify** | **Label** |

All use the same pattern: LoRA + Accelerate + multi-GPU.

| Platform | GPUs | Cost |
|----------|------|------|
| **Kaggle** | 2x T4 | Free (30h/week) |
| **Colab** | 1x T4 | Free |